# Measuring a regression model's error

Every notebook so far has summarised a regression model's errors in one number, usually the mean absolute error of [linear regression](../algorithms/linear-models/linear_regression.ipynb). One number is necessary to compare models, but there are several to choose from, and each answers a different question: how far off the predictions are on average, how bad the worst of them are, how much better than knowing nothing, how far off in proportion to the value. Choosing one is choosing which question matters.

## A model to measure

The gradient boosting model of [Gradient boosting](../algorithms/tree-based/gradient_boosting.ipynb), on California housing, trained on the training rows and measured on the test rows. Values are converted to dollars:

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True).frame
X = housing.drop(columns="MedHouseVal")
y = 100_000 * housing["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=42).fit(X_train, y_train)
y_predicted = model.predict(X_test)

## MAE and RMSE

The **mean absolute error** is the average distance between prediction and true value. The **root mean squared error** squares each error, averages, and takes the root to return to the target's units:

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i| \qquad \text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

{"MAE ($)": mean_absolute_error(y_test, y_predicted), "RMSE ($)": root_mean_squared_error(y_test, y_predicted)}

{'MAE ($)': 31105.521289073844, 'RMSE ($)': 46539.919537971495}

Both are in dollars, yet they differ, because squaring gives large errors more weight. Two sets of predictions for the same five houses, in thousands of dollars, make the difference visible: in the first, every prediction is off by 10; in the second, four are exact and one is off by 50:

In [ ]:
import numpy as np
import pandas as pd

true_values = np.array([200, 150, 300, 250, 180])
pd.DataFrame({
    name: {"MAE": mean_absolute_error(true_values, true_values + errors), "RMSE": root_mean_squared_error(true_values, true_values + errors)}
    for name, errors in {
        "every one off by 10": np.array([10, -10, 10, -10, 10]),
        "one off by 50": np.array([0, 0, 0, 0, 50]),
    }.items()
}).T.round(1)

,MAE,RMSE
every one off by 10,10.0,10.0
one off by 50,10.0,22.4


The same total error gives the same MAE, but the RMSE of the second is more than double: squared, a single error of 50 weighs as much as 25 errors of 10. The RMSE is never smaller than the MAE, and the further apart the two are, the more the error is concentrated in a few large misses. On the model above, an RMSE well above the MAE says the same: most districts are predicted fairly well, and a few very badly.

Which to report depends on what a large error costs. If one error of \$50,000 is worse than five of \$10,000, the RMSE says so; if they are equally bad, the MAE. The MAE is also less affected by a few outliers in the test set.

## The error a model minimises, and the one it is judged by

The function a model minimises while it learns is its **loss**; the measure it is judged by afterwards is its **metric**. In regression they are often the same family: linear regression minimises the squared error, and is often judged by the RMSE. They don't have to be. A loss must suit the learning algorithm (gradient-based methods need it to have a derivative), while a metric must suit the question being asked. In classification the two are usually different: a model learns by minimising the log loss, and is judged by its precision or recall, which can't be optimised directly: they change in steps, with no useful derivative ([Evaluating a classifier](classification_metrics.ipynb)).

## A baseline to compare with

An error of \$30,000 is good or bad only by comparison. The simplest comparison is a model that ignores the features and predicts the same value for every district: `DummyRegressor`. Which constant? The mean of the training targets is the constant with the smallest squared error; the median is the constant with the smallest absolute error. Both, under both measures:

In [ ]:
from sklearn.dummy import DummyRegressor

baselines = {
    "always the mean": DummyRegressor(strategy="mean").fit(X_train, y_train),
    "always the median": DummyRegressor(strategy="median").fit(X_train, y_train),
}
pd.DataFrame({
    name: {"MAE ($)": mean_absolute_error(y_test, baseline.predict(X_test)),
           "RMSE ($)": root_mean_squared_error(y_test, baseline.predict(X_test))}
    for name, baseline in baselines.items()
}).T.round(0)

,MAE ($),RMSE ($)
always the mean,90607.0,114486.0
always the median,87404.0,117312.0


Predicting the same value for every district misses by about \$90,000 on average, and the model by about \$31,000: it removes two thirds of the error of knowing nothing. Each constant wins under the measure it minimises: the median has the lower MAE (\$87,400 against \$90,600), and the mean the lower RMSE. Even a baseline has to be chosen for the measure it will be judged by.

## R²

The **coefficient of determination**, R², builds the comparison with the mean into the measure:

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

the share of the target's variance that the model explains: 1 for a perfect model, 0 for one that always predicts the mean $\bar{y}$ of the data it is measured on. On new data it has no lower limit, and a model worse than the mean has a negative R². The model, the mean, and the model's predictions shifted by \$50,000:

In [ ]:
from sklearn.metrics import r2_score

{
    "model": r2_score(y_test, y_predicted),
    "always the training mean": r2_score(y_test, baselines["always the mean"].predict(X_test)),
    "model, shifted by $50,000": r2_score(y_test, y_predicted + 50_000),
    "model, shifted by $150,000": r2_score(y_test, y_predicted + 150_000),
}

{'model': 0.8347108310026643,
 'always the training mean': -0.00021908714592466794,
 'model, shifted by $50,000': 0.6379795047054088,
 'model, shifted by $150,000': -0.9001639497663978}

The model explains about 83% of the variance of the test values. The training mean scores almost exactly 0, slightly below, because the test rows' own mean differs a little from the training rows'. Shifting every prediction by \$50,000 drops R² to 0.64, and by \$150,000 to -0.90: a model that is systematically off by that much is worse than predicting the mean, and R² says so with a negative number. R² also depends on the data it is measured on: measured only on districts of similar value, the same model would get a lower R², because there would be less variance to explain.

## Errors in proportion to the value

A miss of \$30,000 is small on a district worth \$500,000 and huge on one worth \$60,000. Absolute errors weigh expensive districts more; for many uses, including property valuation, what matters is the error in proportion to the value. The **absolute percentage error** of each prediction is $|y_i - \hat{y}_i| / |y_i|$, and from it come three common summaries:

- **MAPE**, its mean;
- **MdAPE**, its median, less affected by a few huge percentages;
- the **share of predictions within 10%** of the true value (sometimes called PE10), the usual figure for automated valuation models: what proportion of valuations can be trusted to within 10%.

In [ ]:
def percentage_measures(y_true, y_pred):
    percentage_errors = np.abs(y_true - y_pred) / np.abs(y_true)
    return {
        "MAPE (%)": 100 * percentage_errors.mean(),
        "MdAPE (%)": 100 * np.median(percentage_errors),
        "within 10% (%)": 100 * (percentage_errors <= 0.10).mean(),
        "within 20% (%)": 100 * (percentage_errors <= 0.20).mean(),
    }


pd.DataFrame({
    "model": percentage_measures(y_test, y_predicted),
    "always the median": percentage_measures(y_test, baselines["always the median"].predict(X_test)),
}).T.round(1)

,MAPE (%),MdAPE (%),within 10% (%),within 20% (%)
model,18.0,11.8,43.7,70.4
always the median,53.1,36.2,14.8,28.5


On average the model is off by 18% of the value, but half its predictions are within 11.8%: the mean is pulled up by a minority of large percentages. 44% of the districts are valued within 10%, and 70% within 20%; the median baseline manages 15% within 10%. Of all these numbers, "within 10%" is the easiest to explain to whoever will use the predictions: a little under half of these valuations can be trusted to that margin.

Percentage errors have two traps. They are undefined when the true value is 0, and they explode when it is close to 0, so they suit targets that are always well above it, like prices. And they are lopsided: an under-prediction can never be off by more than 100% (as long as no prediction is negative), while an over-prediction can be off by any amount, so a model trained to minimise the MAPE learns to predict low.

## The same errors, by price

Splitting the test districts into five bands of true value shows how the two kinds of measure tell different stories:

In [ ]:
errors = pd.DataFrame({"true value": y_test, "absolute error": np.abs(y_test - y_predicted)})
errors["percentage error"] = 100 * errors["absolute error"] / errors["true value"]
errors["price band"] = pd.qcut(errors["true value"], 5, labels=["cheapest fifth", "second", "third", "fourth", "most expensive fifth"])
errors.groupby("price band", observed=True)[["absolute error", "percentage error"]].agg(["mean", "median"]).round(0)

absolute error          percentage error       
                               mean   median             mean median
price band                                                          
cheapest fifth              21390.0  15290.0             32.0   20.0
second                      23308.0  16087.0             18.0   13.0
third                       24126.0  17841.0             14.0   10.0
fourth                      32749.0  23662.0             13.0   10.0
most expensive fifth        53949.0  38361.0             14.0   10.0

The absolute error grows with the price: about \$21,000 on average for the cheapest fifth of the districts, and \$54,000 for the most expensive. The percentage error goes the other way: 32% for the cheapest fifth, 18% for the second, and 13% to 14% for the rest. Judged in dollars, the model's problem is the expensive districts; judged in proportion, the cheap ones, where an error of \$20,000 is a third of the value. Neither view is wrong, and the use of the predictions decides which one matters. Part of the error of the most expensive fifth comes from the cap at \$500,000 seen in [California housing, step by step](../basic-workflows/california_by_hand.ipynb).

## Which measure answers which question

| Question | Measure |
| --- | --- |
| How far off is a typical prediction, in the target's units? | MAE |
| How far off, counting large misses as worse than several small ones? | RMSE |
| How much better than knowing nothing? | the same measure for a baseline; R² against the mean |
| How far off in proportion to the value? | MAPE, or MdAPE if a few small values make it explode |
| How many predictions can be trusted to within a margin? | the share within 10% (or 5%, or 20%) |
| Where does the model fail? | any of them, by group: [Error analysis: where a model fails](error_analysis.ipynb) |

A report gives more than one, and always next to a baseline. The measure that decides between models has to be chosen before comparing them, so that the choice isn't made by whichever measure favours the model one already prefers.